In [ ]:
# ============================================================
# 环境配置
# - Colab 一般已自带 torch / torchvision
# - 如需手动安装，可取消注释下方 pip 行
# ============================================================

# !pip install torch torchvision matplotlib numpy -q

import importlib
import subprocess
import sys

def ensure_package(pkg):
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["torch", "torchvision", "matplotlib", "numpy"]:
    ensure_package(pkg)

print("Environment check finished.")


In [ ]:
import random
import time
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

import torchvision.models as models

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

plt.rcParams["figure.dpi"] = 120
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False


# TSN 代码实战：学习实现 vs 工程实现

基于论文 *Temporal Segment Networks: Towards Good Practices for Deep Action Recognition*（Wang et al., ECCV 2016），
用一个轻量 toy video classification 任务演示 TSN 的核心思想。

本 Notebook 包含两条并行路径，使用**同一任务**与**同一数据**：

| | 学习路径 (Learning) | 工程路径 (Engineering) |
|---|---|---|
| 目标 | 理解 TSN 的内部机制 | 学会用成熟 backbone 组装 TSN baseline |
| 实现方式 | 手写 segment sampling + shared CNN + consensus | `torchvision` `resnet18` + TSN wrapper |
| 代码量 | 中等 | 更少 |
| 适合场景 | 面试准备、原理掌握 | 简单工程、快速验证 |

> 这不是论文 benchmark 复现，而是一份“能学懂、能讲清、能面试复述”的 TSN 教学 notebook。


## 1. 论文背景与任务定位

### 1.1 TSN 解决了什么问题？

单帧 CNN 往往只能看见非常短的时间窗口；如果直接处理完整长视频，计算与存储代价又很高。
TSN 的思路非常直接：

1. 将长视频划分成 `K` 个时间段（segments）
2. 每段只采样一个代表帧（或 snippet）
3. 用**共享参数**的 backbone 提取每段特征
4. 用 **Segmental Consensus** 将多段信息融合为视频级表示

它的优势是：

- 能覆盖比单帧 CNN 更长的时间范围
- 比 3D CNN / Video Transformer 更轻量
- 非常适合作为视频理解的入门基线

### 1.2 本 Notebook 的范围

本 Notebook 只做：

- 单流（single-stream）TSN
- toy video classification 任务
- 学习路径 + 简单工程路径
- Cross-modality Pre-training / Partial BN 的思路讲解

本 Notebook 不做：

- 真实 UCF101 / Kinetics benchmark 复现
- 光流抽取工程
- 完整双流训练
- 工业部署


## 2. 最小必要理论

### 2.1 TSN 的最小数据流

给定一个视频，TSN 的最小流程可以概括为：

1. 把视频划分成 `K` 个时间段
2. 从每一段采样一个代表帧
3. 用共享参数 backbone 提取每段特征
4. 对 `K` 段特征做 consensus（如 average / max）
5. 再做视频级分类

### 2.2 训练技巧

- **Cross-modality Pre-training**：把 RGB 预训练模型的第一层卷积权重，迁移到光流分支
- **Partial BN**：只保留第一层 BN 可更新，冻结其余 BN，提升小数据集微调稳定性

### 2.3 组件映射表

| 论文组件 | 学习路径覆盖 | 工程路径对应 | 状态 |
|---|---|---|---|
| Segment-based Sampling | `sample_segment_indices()` | 同一采样逻辑复用 | Runnable |
| Shared Backbone | `SimpleCNNBackbone` | `torchvision.models.resnet18` | Runnable |
| Segmental Consensus | 手写 `avg / max` | 同一逻辑复用 | Runnable |
| Cross-modality Pre-training | `cross_modality_init()` 演示 | 第一层权重迁移思路 | Illustrative |
| Partial BN | 原理说明 | `freeze_partial_bn()` | Runnable |
| Two-stream Late Fusion | Markdown 解释 | Markdown 解释 | Explain-only |


## 3. 数据准备

真实视频数据集太大，不适合拿来做一份“快速理解 TSN”的教学 notebook。
这里构造一个轻量 toy dataset：

- 5 个动作类别：`left / right / up / down / stationary`
- 每条视频长度随机变化
- 每帧大小为 `64 x 64`
- 目标小块既有**轨迹差异**，也有**轻微外观差异**

目的不是追求真实 benchmark，而是让 notebook 在 Colab 中几分钟内跑完整流程。


In [ ]:
class SyntheticTSNVideoDataset(Dataset):
    """
    轻量合成视频数据集：
    - 每条视频长度 T 随机
    - 每个类别对应一种运动模式
    - 同时加入轻微外观差异，帮助模型在 toy 任务上更稳定收敛
    """
    ACTION_NAMES = ["left", "right", "up", "down", "stationary"]
    OFFSETS = {
        0: (-2, 0),
        1: (2, 0),
        2: (0, -2),
        3: (0, 2),
        4: (0, 0),
    }

    def __init__(
        self,
        n_samples=200,
        min_frames=12,
        max_frames=24,
        canvas_size=64,
        patch_size=10,
        seed=42,
    ):
        super().__init__()
        self.n_samples = n_samples
        self.min_frames = min_frames
        self.max_frames = max_frames
        self.canvas_size = canvas_size
        self.patch_size = patch_size
        self.rng = np.random.RandomState(seed)

        self.templates = self._build_templates(patch_size)
        self.videos = []
        self.labels = []

        for i in range(n_samples):
            label = i % len(self.ACTION_NAMES)
            video = self._make_video(label)
            self.videos.append(video)
            self.labels.append(label)

    def _build_templates(self, size):
        templates = []
        for label in range(len(self.ACTION_NAMES)):
            patch = np.zeros((size, size), dtype=np.float32)
            patch[1:-1, 1:-1] = 0.25
            patch[size // 2 - 1 : size // 2 + 1, :] += 0.30
            patch[:, size // 2 - 1 : size // 2 + 1] += 0.30

            if label == 0:
                patch[:, 1:3] += 0.25
            elif label == 1:
                patch[:, -3:-1] += 0.25
            elif label == 2:
                patch[1:3, :] += 0.25
            elif label == 3:
                patch[-3:-1, :] += 0.25
            else:
                patch[2:-2, 2:-2] += 0.15

            patch = np.clip(patch, 0.0, 1.0)
            templates.append(patch.astype(np.float32))
        return templates

    def _valid_start_range(self, max_pos, delta, num_steps):
        if delta >= 0:
            low = 0
            high = max_pos - delta * (num_steps - 1)
        else:
            low = -delta * (num_steps - 1)
            high = max_pos
        return int(low), int(high)

    def _make_video(self, label):
        num_frames = self.rng.randint(self.min_frames, self.max_frames + 1)
        dx, dy = self.OFFSETS[label]
        max_pos = self.canvas_size - self.patch_size

        x_low, x_high = self._valid_start_range(max_pos, dx, num_frames)
        y_low, y_high = self._valid_start_range(max_pos, dy, num_frames)

        x0 = self.rng.randint(x_low, x_high + 1)
        y0 = self.rng.randint(y_low, y_high + 1)

        amplitude = float(self.rng.uniform(0.85, 1.15))
        template = self.templates[label] * amplitude

        frames = []
        for t in range(num_frames):
            x = x0 + dx * t
            y = y0 + dy * t

            canvas = np.zeros((self.canvas_size, self.canvas_size), dtype=np.float32)
            canvas[y : y + self.patch_size, x : x + self.patch_size] = template
            canvas += self.rng.normal(0.0, 0.02, size=canvas.shape).astype(np.float32)
            canvas = np.clip(canvas, 0.0, 1.0)
            frames.append(torch.from_numpy(canvas))

        return torch.stack(frames, dim=0)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.videos[idx], int(self.labels[idx])


dataset = SyntheticTSNVideoDataset(n_samples=200, seed=42)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
split_generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=split_generator)

def collate_videos(batch):
    videos, labels = zip(*batch)
    return list(videos), torch.tensor(labels, dtype=torch.long)

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_videos,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_videos,
)

ACTION_NAMES = dataset.ACTION_NAMES

videos_batch, labels_batch = next(iter(train_loader))
print(f"Train videos in batch: {len(videos_batch)}")
print(f"First video shape: {videos_batch[0].shape}")
print(f"First labels: {[ACTION_NAMES[x] for x in labels_batch[:5].tolist()]}")


In [ ]:
def show_video_frames(video, label_name, n_show=6):
    indices = np.linspace(0, video.shape[0] - 1, n_show, dtype=int)
    fig, axes = plt.subplots(1, n_show, figsize=(2.0 * n_show, 2.2))
    for ax, idx in zip(axes, indices):
        ax.imshow(video[idx].cpu(), cmap="gray", vmin=0.0, vmax=1.0)
        ax.set_title(f"t={idx}")
        ax.axis("off")
    fig.suptitle(f"label={label_name}", fontsize=12)
    plt.tight_layout()
    plt.show()

sample_video, sample_label = dataset[0]
show_video_frames(sample_video, ACTION_NAMES[sample_label], n_show=6)


In [ ]:
K_SEGMENTS = 3
NUM_CLASSES = len(ACTION_NAMES)
FEATURE_DIM = 128
DROPOUT = 0.5

LEARNING_EPOCHS = 5
ENGINEERING_EPOCHS = 2

LEARNING_LR = 1e-3
ENGINEERING_LR = 3e-4

print(
    f"K_SEGMENTS={K_SEGMENTS}, NUM_CLASSES={NUM_CLASSES}, "
    f"LEARNING_EPOCHS={LEARNING_EPOCHS}, ENGINEERING_EPOCHS={ENGINEERING_EPOCHS}"
)

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    correct = 0
    total = 0

    for videos, labels in loader:
        labels = labels.to(device)
        logits = model(videos)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return correct / total

def train_model(model, train_loader, val_loader, epochs, lr, partial_bn_fn=None, verbose_label="Model"):
    model.to(device)
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    criterion = nn.CrossEntropyLoss()

    history = {"train_loss": [], "val_acc": []}

    for epoch in range(epochs):
        model.train()
        if partial_bn_fn is not None:
            partial_bn_fn(model)

        running_loss = 0.0
        sample_count = 0
        t0 = time.time()

        for videos, labels in train_loader:
            labels = labels.to(device)
            logits = model(videos)
            loss = criterion(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            sample_count += batch_size

        train_loss = running_loss / sample_count
        val_acc = evaluate_model(model, val_loader)
        elapsed = time.time() - t0

        history["train_loss"].append(train_loss)
        history["val_acc"].append(val_acc)

        print(
            f"[{verbose_label}] epoch {epoch + 1:>2}/{epochs} | "
            f"train_loss={train_loss:.4f} | val_acc={val_acc:.2%} | time={elapsed:.1f}s"
        )

    return history

def plot_histories(learning_history, engineering_history):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(learning_history["train_loss"], marker="o", label="Learning path")
    axes[0].plot(engineering_history["train_loss"], marker="s", label="Engineering path")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss")
    axes[0].set_title("Training Loss")
    axes[0].grid(True, alpha=0.3)
    axes[0].legend()

    axes[1].plot(learning_history["val_acc"], marker="o", label="Learning path")
    axes[1].plot(engineering_history["val_acc"], marker="s", label="Engineering path")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy")
    axes[1].set_title("Validation Accuracy")
    axes[1].grid(True, alpha=0.3)
    axes[1].legend()

    plt.tight_layout()
    plt.show()

@torch.no_grad()
def predict_single_video(model, video):
    model.eval()
    logits = model([video])
    pred = logits.argmax(dim=1).item()
    return pred


## 4. 学习路径：从零实现最小 TSN

这一部分对应 **Learning Path**。
目标不是堆大模型，而是把 TSN 的核心机制拆清楚：

1. 按段采样（segment sampling）
2. 共享参数 backbone
3. Segmental Consensus
4. 视频级分类头

注意：TSN 的“训练 vs 推理差异”主要体现在**采样策略**上。


In [ ]:
def sample_segment_indices(num_frames, k_segments, training=True):
    boundaries = np.linspace(0, num_frames, k_segments + 1)
    indices = []

    for seg_id in range(k_segments):
        start = int(boundaries[seg_id])
        end = int(boundaries[seg_id + 1])
        end = max(end, start + 1)

        if training:
            idx = random.randint(start, end - 1)
        else:
            idx = (start + end - 1) // 2

        idx = min(idx, num_frames - 1)
        indices.append(idx)

    return indices

def sample_batch_frames(videos, k_segments, training):
    batch_frames = []
    batch_indices = []

    for video in videos:
        indices = sample_segment_indices(video.shape[0], k_segments, training=training)
        frames = video[indices].unsqueeze(1).float()
        batch_frames.append(frames)
        batch_indices.append(indices)

    frames = torch.stack(batch_frames, dim=0)
    return frames, batch_indices

class SimpleCNNBackbone(nn.Module):
    def __init__(self, in_channels=1, feature_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(128, feature_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

def segmental_consensus(feats, mode="avg"):
    if mode == "avg":
        return feats.mean(dim=1)
    if mode == "max":
        return feats.max(dim=1).values
    raise ValueError(f"Unsupported consensus mode: {mode}")

class TSNLearningModel(nn.Module):
    def __init__(self, num_classes, k_segments=3, feature_dim=128, consensus="avg", dropout=0.5):
        super().__init__()
        self.k_segments = k_segments
        self.consensus = consensus
        self.backbone = SimpleCNNBackbone(in_channels=1, feature_dim=feature_dim)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(feature_dim, num_classes)

    def forward(self, videos, return_indices=False):
        frames, batch_indices = sample_batch_frames(videos, self.k_segments, training=self.training)
        B, K, C, H, W = frames.shape

        frames = frames.view(B * K, C, H, W).to(device)
        feats = self.backbone(frames)
        feats = feats.view(B, K, -1)

        video_feats = segmental_consensus(feats, mode=self.consensus)
        logits = self.classifier(self.dropout(video_feats))

        if return_indices:
            return logits, batch_indices
        return logits

tsn_learning = TSNLearningModel(
    num_classes=NUM_CLASSES,
    k_segments=K_SEGMENTS,
    feature_dim=FEATURE_DIM,
    consensus="avg",
    dropout=DROPOUT,
).to(device)

test_logits = tsn_learning(videos_batch[:4])
print(f"Learning path output shape: {test_logits.shape}")


## 5. 训练 vs 推理：TSN 的关键区别

| 阶段 | 行为 | 目的 |
|---|---|---|
| 训练 | 每段随机采样 | 增加扰动，减少过拟合 |
| 推理 | 每段中心采样 | 保证预测稳定性与可重复性 |

这和 seq2seq 模型里的 teacher forcing 不同。
TSN 的差异点不在解码，而在**时间采样策略**。


In [ ]:
sample_video, sample_label = dataset[0]

tsn_learning.train()
_, train_indices = tsn_learning([sample_video], return_indices=True)

tsn_learning.eval()
_, eval_indices = tsn_learning([sample_video], return_indices=True)

print(f"Train-time indices : {train_indices[0]}")
print(f"Inference indices  : {eval_indices[0]}")

learning_history = train_model(
    tsn_learning,
    train_loader,
    val_loader,
    epochs=LEARNING_EPOCHS,
    lr=LEARNING_LR,
    partial_bn_fn=None,
    verbose_label="Learning",
)

learning_val_acc = evaluate_model(tsn_learning, val_loader)
print(f"Learning path final val acc: {learning_val_acc:.2%}")


## 6. 工程路径：从学习到简单工程

这一部分对应 **Engineering Path**。
TSN 并没有像 `transformers` 那样特别成熟的一体化高层工业库，因此更合理的工程表达是：

- backbone 用成熟库（这里用 `torchvision.models.resnet18`）
- TSN 的分段采样与 consensus 逻辑自己封装
- 引入工程技巧：`Partial BN`
- 补充一个说明性演示：`Cross-modality Pre-training`

为了保留 `resnet18` 的预训练第一层卷积，这里不把输入改成单通道，而是把灰度帧**复制成 3 通道**后再送入 backbone。


In [ ]:
def freeze_partial_bn(backbone):
    bn_count = 0
    for m in backbone.modules():
        if isinstance(m, nn.BatchNorm2d):
            bn_count += 1
            if bn_count > 1:
                m.eval()
                for p in m.parameters():
                    p.requires_grad = False

def cross_modality_init(pretrained_conv, flow_channels=20):
    old_weight = pretrained_conv.weight.detach().cpu()
    mean_weight = old_weight.mean(dim=1, keepdim=True)
    new_weight = mean_weight.repeat(1, flow_channels, 1, 1)

    new_conv = nn.Conv2d(
        in_channels=flow_channels,
        out_channels=old_weight.shape[0],
        kernel_size=pretrained_conv.kernel_size,
        stride=pretrained_conv.stride,
        padding=pretrained_conv.padding,
        bias=(pretrained_conv.bias is not None),
    )
    new_conv.weight.data.copy_(new_weight)

    if pretrained_conv.bias is not None:
        new_conv.bias.data.copy_(pretrained_conv.bias.detach().cpu())

    return new_conv

class TSNEngineeringModel(nn.Module):
    def __init__(self, num_classes, k_segments=3, consensus="avg", dropout=0.2, pretrained=True):
        super().__init__()
        self.k_segments = k_segments
        self.consensus = consensus

        try:
            weights = models.ResNet18_Weights.DEFAULT if pretrained else None
            backbone = models.resnet18(weights=weights)
            self.pretrained_loaded = pretrained
        except Exception as e:
            print(f"Pretrained weights unavailable, fallback to random init: {e}")
            backbone = models.resnet18(weights=None)
            self.pretrained_loaded = False

        feat_dim = backbone.fc.in_features
        backbone.fc = nn.Identity()

        self.backbone = backbone
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(feat_dim, num_classes)

    def forward(self, videos, return_indices=False):
        frames, batch_indices = sample_batch_frames(videos, self.k_segments, training=self.training)
        B, K, C, H, W = frames.shape

        frames = frames.view(B * K, 1, H, W)
        frames = frames.repeat(1, 3, 1, 1)
        frames = F.interpolate(frames, size=(224, 224), mode="bilinear", align_corners=False)
        frames = frames.to(device)

        feats = self.backbone(frames)
        feats = feats.view(B, K, -1)

        video_feats = segmental_consensus(feats, mode=self.consensus)
        logits = self.classifier(self.dropout(video_feats))

        if return_indices:
            return logits, batch_indices
        return logits


In [ ]:
tsn_engineering = TSNEngineeringModel(
    num_classes=NUM_CLASSES,
    k_segments=K_SEGMENTS,
    consensus="avg",
    dropout=0.2,
    pretrained=True,
).to(device)

print(f"Engineering path pretrained loaded: {tsn_engineering.pretrained_loaded}")

freeze_partial_bn(tsn_engineering.backbone)
flow_conv1 = cross_modality_init(tsn_engineering.backbone.conv1, flow_channels=20)
print(f"Cross-modality conv weight shape: {tuple(flow_conv1.weight.shape)}")

engineering_history = train_model(
    tsn_engineering,
    train_loader,
    val_loader,
    epochs=ENGINEERING_EPOCHS,
    lr=ENGINEERING_LR,
    partial_bn_fn=lambda model: freeze_partial_bn(model.backbone),
    verbose_label="Engineering",
)

engineering_val_acc = evaluate_model(tsn_engineering, val_loader)
print(f"Engineering path final val acc: {engineering_val_acc:.2%}")


## 7. 学习路径 vs 工程路径对比

### 7.1 六维对比

| 对比维度 | 学习路径 | 工程路径 |
|---|---|---|
| 教育价值 | 理解 TSN 机制本身 | 理解如何复用成熟 backbone |
| 代码量 | 中等 | 更少 |
| 灵活性 | 高，可改任何组件 | 中，受 backbone 结构约束 |
| 稳定性 | 中，完全依赖手写实现 | 高，backbone 更成熟 |
| 工业适配度 | 研究 / 教学 / 面试 | baseline / 快速实验 / 简单工程 |
| 适用场景 | 学原理、讲原理、面试复述 | 从学习走向简单工程实践 |

### 7.2 组件对应关系

| 学习路径实现 | 工程路径对应 | 说明 |
|---|---|---|
| `SimpleCNNBackbone` | `resnet18` | 同样做单帧表征，只是抽象层级更高 |
| `sample_segment_indices()` | 同一采样函数复用 | TSN 核心策略本身没有变 |
| `segmental_consensus()` | 同一 consensus 复用 | TSN 关键逻辑依然保留 |
| 手写训练逻辑 | `Partial BN` + 预训练 backbone | 工程上更强调稳定与复用 |
| `cross_modality_init()` 演示 | 光流分支初始化思路 | 属于工程 trick，不是主任务主线 |

### 7.3 路径边界

**学习路径得到什么：**

- 看懂 TSN 的基本数据流：分段采样 -> 共享 backbone -> consensus -> 分类
- 能说清训练与推理的采样差异
- 能在面试里解释 TSN 为什么比单帧 CNN 更合理、比 3D CNN 更轻

**工程路径得到什么：**

- 能把 TSN 的核心思路和成熟视觉 backbone 组合起来
- 能理解 `Partial BN` 这类迁移训练技巧
- 能把 `Cross-modality Pre-training` 作为“工程 trick”讲清楚

**本 Notebook 没覆盖什么：**

- 没有完整双流训练
- 没有真实光流提取
- 没有真实 UCF101 / Kinetics benchmark
- 没有部署与性能优化


In [ ]:
plot_histories(learning_history, engineering_history)

print(f"Learning path validation accuracy   : {learning_val_acc:.2%}")
print(f"Engineering path validation accuracy: {engineering_val_acc:.2%}")

print("\nSample predictions on validation set:")
for idx in range(5):
    video, label = val_dataset[idx]
    pred_l = predict_single_video(tsn_learning, video)
    pred_e = predict_single_video(tsn_engineering, video)

    print(
        f"sample {idx:>2} | label={ACTION_NAMES[label]:<10} | "
        f"learning={ACTION_NAMES[pred_l]:<10} | engineering={ACTION_NAMES[pred_e]:<10}"
    )

video_demo, label_demo = val_dataset[0]
show_video_frames(video_demo, f"gt={ACTION_NAMES[label_demo]}", n_show=6)


# 8. 面试高频问题

## Q1. TSN 为什么比单帧 CNN 更合理？

单帧 CNN 只看一个时刻，容易漏掉动作的时间范围。
TSN 把视频分成多个时间段，各段采一个代表帧，再融合成视频级表示，因此能用很低代价覆盖更长时间范围。

---

## Q2. 为什么 TSN 用的是“分段采样”，而不是连续 clip？

因为 TSN 的目标不是像 3D CNN 那样建模密集局部运动，而是用稀疏采样获得整个视频的全局时间覆盖。
它牺牲了局部时序细节，换来了更低计算成本和更长时间感受野。

---

## Q3. Shared Backbone 的意义是什么？

每个 segment 都来自同一个视频，只是时间位置不同。
如果 backbone 不共享，段间特征就不在同一个特征空间里，consensus 的意义会被削弱。
共享参数既减少参数量，也保证了特征可比较。

---

## Q4. Average Consensus 和 Max Consensus 怎么选？

- `Average`：更稳定，适合作为默认配置
- `Max`：更强调关键帧，但也更容易受噪声影响

在教学 notebook 里，`Average` 更适合作为主线，因为更容易解释，也更稳。

---

## Q5. TSN 的训练和推理差异在哪里？

主要在**采样策略**：

- 训练：每段随机采样，增加扰动，相当于一种时间维数据增强
- 推理：每段中心采样，保证结果稳定、可重复

---

## Q6. Partial BN 为什么有效？

在小数据集上，BN 的 running statistics 很容易漂。
TSN 论文中的做法是只保留第一层 BN 继续适应输入分布变化，冻结后续 BN，减少小数据集微调时的不稳定性。

---

## Q7. Cross-modality Pre-training 的本质是什么？

本质是把 RGB 分支预训练好的第一层卷积权重迁移到光流分支。
常见做法是先对 RGB 三通道求平均，再复制到多个光流通道。
它不是 TSN 数学定义的一部分，而是一个很重要的工程训练技巧。

---

## Q8. TSN 和 3D CNN / Video Transformer 的差别是什么？

- **TSN**：显式做时间稀疏采样，再融合
- **3D CNN**：在卷积核内部直接建模时空局部关系
- **Video Transformer**：用自注意力显式建模更长范围时空依赖

如果数据和算力有限，TSN 常常是一个很好的起点；
如果追求更强时序建模能力，则通常会走向 3D CNN 或 Video Transformer。

---

# 9. 面试速答提纲 + 项目表达

## 9.1 面试速答提纲

| 问题 | 一句话回答 |
|---|---|
| TSN 核心思想是什么？ | 长视频分段采样，共享 backbone，再做共识融合 |
| 为什么比单帧 CNN 更强？ | 它覆盖了更长时间范围，而不是只看一个时刻 |
| 为什么比 3D CNN 更轻？ | 它不做时空卷积，只对少量代表帧做 2D 特征提取 |
| 训练和推理差异在哪？ | 训练随机采样，推理中心采样 |
| Partial BN 是什么？ | 冻结大部分 BN，减少小数据集统计漂移 |
| Cross-modality Pre-training 是什么？ | 把 RGB 预训练第一层迁移到光流分支 |
| 工程路径为什么用 ResNet18？ | 成熟、稳定、容易复用，适合做简单工程 baseline |
| 这份 notebook 的限制是什么？ | 单流、toy data、不是 benchmark 复现 |

## 9.2 项目表达模板

如果面试官问：“你做过什么相关项目？”
你可以这样回答：

> 我做过一个 TSN 教学型 baseline。
> 学习路径里，我从零实现了 segment sampling、shared backbone 和 segmental consensus，理解了 TSN 为什么能在不做 3D 卷积的情况下覆盖更长视频时间范围。
> 工程路径里，我用 `torchvision` 的 `resnet18` 作为成熟 backbone，再自己组装 TSN 的采样与融合逻辑，并加入 `Partial BN` 这种迁移训练技巧。
> 这个项目让我既能讲清 TSN 的原理，也能说清它和 3D CNN、Video Transformer 在建模方式、算力成本和适用场景上的差异。

## 9.3 延伸阅读与对比

| 模型 | 核心思想 | 计算成本 | 适用场景 |
|---|---|---|---|
| TSN | 分段采样 + 共识 | 低 | 入门视频理解、轻量 baseline |
| 3D CNN | 时空卷积 | 中高 | 更强局部时序建模 |
| Video Transformer | 时空注意力 | 高 | 大模型、大数据、更强表达能力 |

## 9.4 进阶探索方向

1. 把当前单流 TSN 扩展成双流 TSN（RGB + optical flow）
2. 把 `avg` consensus 换成更复杂的 temporal modeling 模块
3. 换成真实数据集（如 UCF101）做小规模复现
4. 把 TSN 和 3D CNN / Video Transformer 做系统对比
